# Airbnb Data Analysis - Full EDA
Data reading + general descriptive statistics + Univariate analysis + Bivariate analysis + Multivariate analysis + Correlation Matrix

All plots use **Seaborn** with a unified, eye-friendly color palette.


## 0) Imports and General Theme Setup


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="talk")

MAIN_PALETTE = "crest"
CAT_PALETTE = "Set2"
ACCENT_COLOR = "#4C72B0"
DIVERGING_PALETTE = "coolwarm"

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["figure.dpi"] = 110

%matplotlib inline


## 1) Data Reading


In [ ]:
DATA_PATH = "airbnb_combined.csv"
df = pd.read_csv(DATA_PATH)

print(f"Data shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


In [ ]:
print("Column data types (dtypes):")
df.dtypes


In [ ]:
print("Number of missing values in each column:")
df.isnull().sum()


In [ ]:
print(f"Number of duplicate rows (Duplicates): {df.duplicated().sum()}")


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
binary_like = [c for c in numeric_cols if df[c].dropna().nunique() <= 2]
numeric_cols_true = [c for c in numeric_cols if c not in binary_like]
categorical_cols = df.select_dtypes(include=["object", "bool", "category"]).columns.tolist()

print("Continuous numeric columns:", numeric_cols_true)
print("\nCategorical / boolean columns:", categorical_cols)
print("\nBinary numeric columns (0/1):", binary_like)


## 2) General Descriptive Statistics


In [ ]:
desc_numeric = df[numeric_cols_true].describe().T
desc_numeric["skewness"] = df[numeric_cols_true].skew()
desc_numeric["kurtosis"] = df[numeric_cols_true].kurt()
desc_numeric

In [ ]:
desc_cat = df[categorical_cols].describe().T
desc_cat

## 3) Univariate Analysis


### 3.1 Distribution of Continuous Numeric Variables (Histogram + KDE)


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols_true) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows))
axes = axes.flatten()
for i, col in enumerate(numeric_cols_true):
    sns.histplot(df[col], kde=True, ax=axes[i], color=ACCENT_COLOR, bins=40)
    axes[i].set_title(f"Distribution of {col}")
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
fig.suptitle("Univariate Analysis - Distribution of Numeric Variables", y=1.02, fontsize=18)
plt.tight_layout()
plt.show()


### 3.2 Boxplot for Outlier Detection (Outliers)


In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(numeric_cols_true):
    sns.boxplot(x=df[col], ax=axes[i], color=sns.color_palette(MAIN_PALETTE, 1)[0])
    axes[i].set_title(f"Boxplot - {col}")
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
fig.suptitle("Univariate Analysis - Outlier Detection (Boxplots)", y=1.02, fontsize=18)
plt.tight_layout()
plt.show()

### 3.3 Distribution of Categorical Variables (Countplot)


In [ ]:
cat_to_plot = ["room_type", "city", "day_type", "host_is_superhost", "room_shared", "room_private"]
cat_to_plot = [c for c in cat_to_plot if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
axes = axes.flatten()
for i, col in enumerate(cat_to_plot):
    order = df[col].value_counts().index
    sns.countplot(y=df[col], order=order, ax=axes[i], palette=CAT_PALETTE)
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel("Count")
fig.suptitle("Univariate Analysis - Categorical Variables Distribution", y=1.02, fontsize=18)
plt.tight_layout()
plt.show()


### 3.4 Binary Columns (Flags: multi, biz)


In [ ]:
if binary_like:
    fig, axes = plt.subplots(1, len(binary_like), figsize=(6 * len(binary_like), 5))
    if len(binary_like) == 1:
        axes = [axes]
    for i, col in enumerate(binary_like):
        sns.countplot(x=df[col], ax=axes[i], palette=CAT_PALETTE)
        axes[i].set_title(f"Distribution of {col}")
    fig.suptitle("Univariate Analysis - Binary Flags", y=1.05, fontsize=16)
    plt.tight_layout()
    plt.show()


## 4) Bivariate Analysis


In [ ]:
TARGET = "realSum"


### 4.1 Average Price by Room Type


In [ ]:
plt.figure(figsize=(10, 6))
order = df.groupby("room_type")[TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="room_type", y=TARGET, order=order, palette=CAT_PALETTE, showfliers=False)
plt.title("Bivariate: Price Distribution by Room Type")
plt.ylabel("Price (realSum)")
plt.tight_layout()
plt.show()

### 4.2 Average Price by City


In [ ]:
plt.figure(figsize=(14, 7))
order_city = df.groupby("city")[TARGET].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="city", y=TARGET, order=order_city, palette=MAIN_PALETTE, showfliers=False)
plt.title("Bivariate: Price Distribution by City")
plt.xticks(rotation=30)
plt.ylabel("Price (realSum)")
plt.tight_layout()
plt.show()

### 4.3 Price by Day Type (Weekday/Weekend)


In [ ]:
plt.figure(figsize=(8, 6))
sns.violinplot(data=df, x="day_type", y=TARGET, palette=CAT_PALETTE, cut=0)
plt.title("Bivariate: Price Distribution by Day Type")
plt.tight_layout()
plt.show()

### 4.4 Relationship Between Distance from Center and Price


In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(data=df.sample(min(4000, len(df)), random_state=42),
                 x="dist", y=TARGET, alpha=0.4, color=ACCENT_COLOR)
plt.title("Bivariate: Price vs Distance from City Center")
plt.tight_layout()
plt.show()

### 4.5 Relationship Between Cleanliness Rating and Price


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="cleanliness_rating", y=TARGET, palette=MAIN_PALETTE, showfliers=False)
plt.title("Bivariate: Price by Cleanliness Rating")
plt.tight_layout()
plt.show()

### 4.6 Price by Superhost Status


In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="host_is_superhost", y=TARGET, palette=CAT_PALETTE, showfliers=False)
plt.title("Bivariate: Price by Superhost Status")
plt.tight_layout()
plt.show()

### 4.7 Number of Listings by City and Room Type


In [ ]:
plt.figure(figsize=(15, 7))
sns.countplot(data=df, x="city", hue="room_type", order=order_city, palette=CAT_PALETTE)
plt.title("Bivariate: Room Type Count by City")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5) Correlation Matrix


In [ ]:
corr_cols = numeric_cols_true + binary_like
corr_matrix = df[corr_cols].corr(numeric_only=True)
corr_matrix.round(2)

In [ ]:
plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap=DIVERGING_PALETTE, center=0, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix - Numeric Variables")
plt.tight_layout()
plt.show()

### Strongest Factors Correlated with Price (Target)


In [ ]:
price_corr = corr_matrix[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
price_corr

In [ ]:
plt.figure(figsize=(9, 7))
sns.barplot(x=price_corr.values, y=price_corr.index, palette=MAIN_PALETTE)
plt.axvline(0, color="grey", linewidth=1)
plt.title(f"Correlation of Features with {TARGET}")
plt.xlabel("Correlation Coefficient")
plt.tight_layout()
plt.show()

## 6) Multivariate Analysis


### 6.1 Pairplot of Top Variables Correlated with Price


In [ ]:
top_features = price_corr.abs().sort_values(ascending=False).head(4).index.tolist()
pairplot_cols = [TARGET] + top_features
sample_df = df.sample(min(3000, len(df)), random_state=42)

g = sns.pairplot(sample_df[pairplot_cols + ["room_type"]],
                  hue="room_type", palette=CAT_PALETTE,
                  plot_kws={"alpha": 0.5, "s": 25}, diag_kind="kde", height=2.6)
g.fig.suptitle("Multivariate: Pairplot of Top Correlated Features (colored by Room Type)", y=1.02)
plt.show()

### 6.2 Price by City and Room Type Combined


In [ ]:
plt.figure(figsize=(16, 8))
sns.boxplot(data=df, x="city", y=TARGET, hue="room_type",
            order=order_city, palette=CAT_PALETTE, showfliers=False)
plt.title("Multivariate: Price by City and Room Type")
plt.xticks(rotation=30)
plt.legend(title="Room Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 6.3 Price vs Distance, Colored by City (Bubble Size = Person Capacity)


In [ ]:
plt.figure(figsize=(12, 8))
sample_df2 = df.sample(min(3000, len(df)), random_state=42)
sns.scatterplot(data=sample_df2, x="dist", y=TARGET, hue="city",
                 size="person_capacity", sizes=(20, 200), alpha=0.55,
                 palette="tab10")
plt.title("Multivariate: Price vs Distance colored by City (size = capacity)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

### 6.4 Heatmap: Median Price by City and Room Type


In [ ]:
pivot_table = df.pivot_table(values=TARGET, index="city", columns="room_type", aggfunc="median")
plt.figure(figsize=(9, 8))
sns.heatmap(pivot_table, annot=True, fmt=".0f", cmap=MAIN_PALETTE, linewidths=0.5)
plt.title("Multivariate: Median Price Heatmap (City x Room Type)")
plt.tight_layout()
plt.show()

### 6.5 Price Distribution by Day Type per City (FacetGrid)


In [ ]:
g2 = sns.FacetGrid(df, col="city", col_wrap=5, height=3.2, sharex=False, sharey=False)
g2.map_dataframe(sns.kdeplot, x=TARGET, hue="day_type", fill=True,
                  palette=CAT_PALETTE, alpha=0.4, common_norm=False)
g2.set_titles("{col_name}")
g2.fig.suptitle("Multivariate: Price Density by Day Type across Cities", y=1.02)
g2.add_legend()
plt.show()

## 7) Conclusion

- The strongest factors correlated with the nightly price (`realSum`) are: `attr_index_norm` (proximity to tourist attractions), `bedrooms`, `lat`.
- Cleanliness rating and overall guest rating show almost no linear correlation with price.
- There are clear price differences between cities and room types (Entire home/apt is usually more expensive).
